In [18]:
secfiles = [
    "sections-shumei-sexy-241118.json",
    "pachong2/sections-shumei-porn1103.json",
    "pachong2/sections-shumei-politics1103.json",
    "sections-opencompass.json",
    "sections-data-tool2-shumei.json",
    "sections-shumei-seqingyouxi.json",
    "sections-data-tool-shumei.json",
    "sections-nsfw.json",
    "sections-nsfw-shumei.json",
    "sections-safety240722.json",
]

import json

secdata = {}
seccounts = {}

for sf in secfiles:
    data = json.load(open(sf))
    secdata[sf] = data

    counts = seccounts[sf] = {}

    for k, v in data["count"].items():
        counts[k] = v

from pprint import pprint

pprint(seccounts)

{'pachong2/sections-shumei-politics1103.json': {'normal': 58105,
                                                'politics': 11235,
                                                'porn': 229},
 'pachong2/sections-shumei-porn1103.json': {'normal': 320, 'porn': 3982},
 'sections-data-tool-shumei.json': {'ban': 4886,
                                    'blackandwhitelist': 21,
                                    'minor': 195,
                                    'normal': 118249,
                                    'politics-1001': 7374,
                                    'politics-1002': 11961,
                                    'porn-1001': 1003,
                                    'porn-1002': 3023,
                                    'sexy': 0,
                                    'star': 347,
                                    'violence': 923},
 'sections-data-tool2-shumei.json': {'normal': 181292,
                                     'politics-1001': 23866,
                       

In [19]:
import random
from copy import deepcopy

collects = {
    "sections-shumei-sexy-241118.json": {
        "normal": {"train": 0.3, "val": 100, "label": "normal"},
        "porn": {"train": 0.5, "val": 100, "label": "porn"},
    },
    # 1103
    "pachong2/sections-shumei-politics1103.json": {
        "normal": {"train": 0.3, "val": 0, "label": "normal"},
        "politics": {"train": 1, "val": 0, "label": "politics"},
    },
    "pachong2/sections-shumei-porn1103.json": {
        "normal": {"train": 2, "val": 0, "label": "normal"},
        "porn": {"train": 1, "val": 0, "label": "porn"},
    },
    "sections-opencompass.json": {
        "force-all-normal": {"train": 1, "val": 0, "label": "normal"},
    },
    # 0930
    "sections-data-tool2-shumei.json": {
        "normal": {"train": 0.3, "val": 200, "label": "normal"},
        # "politics-1001": {"train": 1.0, "val": 500, "label": "politics"},
        "politics-1002": {"train": 1.0, "val": 500, "label": "politics"},
        # "porn-1002": {"train": 1.0, "val": 50, "label": "porn"},
    },
    # "sections-shumei-seqingyouxi.json": {
    #     "normal": {"train": 1.0, "val": 50, "label": "normal"},
    #     "porn": {"train": 5.0, "val": 50, "label": "porn"},
    # },
    # 以前
    "sections-data-tool-shumei.json": {
        "normal": {"train": 0.3, "val": 200, "label": "normal"},
        # "politics-1001": {"train": 1.0, "val": 500, "label": "politics"},
        "politics-1002": {"train": 1.0, "val": 500, "label": "politics"},
        "porn-1002": {"train": 1.0, "val": 350, "label": "porn"},
    },
    "sections-nsfw-shumei.json": {
        "normal": {"train": 1.0, "val": 100, "label": "normal"},
        "porn": {"train": 1.0, "val": 100, "label": "porn"},
    },
    "sections-nsfw.json": {
        "drawings": {"train": 0.5, "val": 100, "label": "normal"},
        "neutral": {"train": 0.5, "val": 100, "label": "normal"},
    },
    "sections-safety240722.json": {
        "20240322_8k_nsfw_dup_1507": {"train": 0.8, "val": 50, "label": "porn"},
        "20240329_163k_nsfw_dup_110k": {"train": 0.8, "val": 50, "label": "porn"},
        "nsfw_23k": {"train": 0.8, "val": 50, "label": "porn"},
        "sq_8k": {"train": 0.8, "val": 50, "label": "porn"},
    },
}

train_counts = {}
val_counts = {}
train_anns = []
val_anns = []
labels = ["normal", "politics", "porn"]
label_map = {"normal": 0, "politics": 1, "porn": 2}

random.seed(0)

for sf, col in collects.items():
    # for each file, collect multiple sections
    counts = seccounts[sf]
    data = secdata[sf]

    for k, v in col.items():
        # collect sections

        label = v["label"]
        val_count = int(v.get("val", 0))
        train_count = int((counts[k] - val_count) * v["train"])

        # stat
        if label in train_counts:
            train_counts[label] += train_count
        else:
            train_counts[label] = train_count
        if label in val_counts:
            val_counts[label] += val_count
        else:
            val_counts[label] = val_count
        if label not in labels:
            labels.append(label)
            label_map[label] = len(labels) - 1
        label_id = label_map[label]

        # collect data
        files = deepcopy(data["files"][k])
        random.shuffle(files)

        # get val first to avoid overlap
        for _ in range(val_count):
            val_anns.append((files.pop(), label_id))

        # sample train
        files = files * int(v["train"] + 1)
        files = files[: int(train_count)]

        train_ann_ = [(f, label_id) for f in files]
        train_anns.extend(train_ann_)

pprint(train_counts)
pprint(val_counts)
pprint(train_anns[:: len(train_anns) // 20])
pprint(val_anns[:: len(val_anns) // 20])

assert set(train_anns) & set(val_anns) == set()


{'normal': 166276, 'politics': 68635, 'porn': 198546}
{'normal': 800, 'politics': 1000, 'porn': 750}
[('sexy-241118/cache4.bp.blogspot.com/-piQNZV-WXFE/WBf1MXtGrdI/AAAAAAAAk9Y/28mGbIet0CwqvC-dImvbCMmz0EI3Ei55ACLcB/s1600/MrCong.com-MyGirl-Vol.049-Pan-Jiaojiao-011.jpg',
  0),
 ('sexy-241118/cachewes.misskon.com/images/2024/08/20/XIUREN-No.6823-Wang-Wan-You-Queen-MissKON.com-027297ecaa8be30ac6e.webp',
  0),
 ('sexy-241118/cachelux.mrcong.com/images/2023/12/02/XIUREN-No.6323-Zhou-Yuxi-Sally-MrCong.com-004.webp',
  2),
 ('sexy-241118/cachewes.misskon.com/images/2024/08/20/XIUREN-No.6822-babe-MissKON.com-050af3ad3099d0cb845.webp',
  2),
 ('sexy-241118/cachewww.ik009.top/uploadfile/202207/20/A7183517872.webp', 2),
 ('sexy-241118/cachewes.misskon.com/images/2024/08/08/XIUREN-No.6753-Jiang-Zhen-Zhen-MissKON.com-07018b093785a087e3b.webp',
  2),
 ('pachong2/Twitter/Ruters0615/1796693140851798293_2.jpg', 1),
 ('oc_data/HallusionBench/VD_figure_1_1_0_0.jpg', 0),
 ('data_tool2/res2/岳昕/google_岳昕/0026

In [20]:
with open('huangfan-shumei-1120-train.txt', 'w') as f:
    for ann in train_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-1120-val.txt', 'w') as f:
    for ann in val_anns:
        f.write(f"{ann[0]} {ann[1]}\n")

with open('huangfan-shumei-1120-labels.txt', 'w') as f:
    for label in labels:
        f.write(f"{label}\n")